# [심화·선택] 09. LangGraph (Advanced) — 제품 탑재형 AI: Router → (Manual / Control / Troubleshoot / Handoff)
**LangGraph로 제품 탑재형 파이프라인(멀티 워크플로우)을 구성해보기**

---

## 🧩 시나리오(가상)
스마트홈/로보틱스/생활가전 탑재형 AI는 "무엇을 해야 하는지"를 먼저 분기해야 합니다.

- 매뉴얼 근거 기반 답변이 필요한가? (manual_info)
- 기기 제어를 실행해야 하는가? (device_control)
- 고장 진단 흐름이 필요한가? (troubleshooting)
- 안전/권한 이슈로 사람 연결이 필요한가? (handoff)

이 노트북은 LangGraph로 위 라우팅 + 각 워크플로우를 **하나의 그래프**로 묶고,
Streaming으로 내부 실행 흐름을 관찰합니다.

> 데이터/도구는 전부 더미이며, 실제 제품/현업 시스템과 무관합니다.

> **사전 조건 (로컬 LLM)**: `03_llama_server_local_llm.ipynb`에서 띄운 **llama-server(http://localhost:8080 · 기본 Qwen3.5-4B / 옵션 Qwen2.5-7B)** 가 실행 중이어야 합니다.
> 서버가 꺼져 있다면 터미널에서 `python local_server.py start 4b` 를 실행하세요. (API 키/비용 없음, 모든 추론은 내 GPU에서 수행)



이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


이 노트북은 6시간 핵심 과정 이후 진행하는 심화 선택 자료입니다.


## 🧭 시작 전에

| 항목 | 내용 |
|---|---|
| 🎯 학습 목표 | 입력 의도에 따라 서로 다른 워크플로우로 분기하는 LLM 앱을 그래프로 설계·실행·관찰한다 (05~08 종합) |
| 📚 사전 지식 | 05번(구조화 출력 → 라우터), 06번(도구·메모리), 07번(근거 태그) |
| 🐍 커널(환경) | `LLM Hands-On (Win CUDA)` |
| 🖥️ GPU 사용 | 공용 llama-server |
| ⏱️ 소요 시간 | 약 50분 |
| ✔️ 실행 전 확인 | llama-server 실행 중 |

> **실행 요령**: 위에서부터 **한 셀씩**(Shift+Enter) 실행하세요. `Run All`은 쓰지 마세요.
> `✅ TODO 셀`은 학생이 채우는 칸이라 그대로 실행하면 일부러 에러가 납니다. 막히면 바로 아래 `✅ 참고 답안` 셀을 보세요.

## 📖 핵심 개념: 왜 그래프로 설계할까?

06번의 `create_agent`는 "LLM ↔ 도구" 반복이라는 **정해진 루프**입니다. 실제 서비스는 더 복잡합니다.
"안전 문제면 즉시 사람에게", "제어 요청이면 상태 확인 후 실행", "고장이면 진단 → 매뉴얼 → 티켓"…
이런 흐름을 **노드(할 일)와 엣지(다음 경로)** 로 직접 설계하는 도구가 **LangGraph**입니다.
(참고: `create_agent`도 내부적으로는 LangGraph 그래프입니다)

```
                ┌─▶ manual_agent ⇄ manual_tools ──────────┐
START ─▶ router ┼─▶ control_agent ⇄ control_tools ────────┼─▶ END
 (의도 분류)     ├─▶ troubleshoot_agent ⇄ troubleshoot_tools┤
                └─▶ handoff (사람 연결) ───────────────────┘
```

| 용어 | 설명 |
|---|---|
| **State** | 노드들이 함께 읽고 쓰는 공유 데이터 (여기서는 `messages`, `intent`) |
| **Node** | 한 단계의 작업을 하는 함수 (LLM 호출, 도구 실행 등) |
| **Edge** | 노드 사이의 고정 경로 |
| **Conditional edge** | State를 보고 다음 노드를 고르는 분기 (`route_from_intent`, `tools_condition`) |
| **ToolNode** | LLM이 요청한 도구를 실행해 주는 미리 만들어진 노드 |
| **Router** | 05번의 구조화 출력으로 의도를 4가지 중 하나로 분류 → 분기 근거 |
| **stream** | 노드가 실행될 때마다 중간 결과를 받아 흐름을 관찰 |

In [ ]:
# !pip -q install -U langgraph langchain langchain-openai pydantic requests


In [ ]:
import os, getpass, re, json
from typing import Annotated, Literal, Optional, List, Dict, Any
from typing_extensions import TypedDict

from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, SystemMessage, AIMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
# 그래프 객체 빈거 StateGraph 인스턴스 만들고 add_edge / add_node 메서드 호출 예정
# START END는 특수 노드. 따로 우리가 정의하지 않아도 됨
from langgraph.graph.message import add_messages
# State 정의에서 대화 히스토리에 Message 객체 누적시키는 함수
from langgraph.prebuilt import ToolNode, tools_condition
# ToolNode는 tool 모음으로 instantiate할 노드 클래스
# AgentNode는? -> 우리가 정의해서 쓸 예정
# tools_condition: 분기 함수 (랭그래프 빌트인)
# 가장 최근 AIMessage를 보고 tool_calls 필드가 True인지 조회
from langgraph.checkpoint.memory import InMemorySaver

# ── 로컬 LLM 연결 (공용 설정: local_llm.py) ──
# 모든 앱 실습(05~09)은 같은 로컬 OpenAI 호환 서버(llama-server)를 사용합니다.
from local_llm import get_chat_model, check_server, LLM_BASE_URL, LLM_MODEL
check_server()   # 서버가 안 떠 있으면 해결 방법을 알려주고 에러를 냅니다.

llm = get_chat_model(temperature=0)   # 로컬 LLM (llama-server)


## 1) State 정의

messages 외에 intent를 상태로 관리합니다.


In [ ]:
class GraphState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    intent: Optional[Literal["manual_info","device_control","troubleshooting","handoff"]]


## 2) Router 노드: 입력 의도 분류(구조화)

LLM에게 intent를 **4값(manual_info/device_control/troubleshooting/handoff)** 중 하나로만 출력하게 하여,
제품 탑재형 환경에서 중요한 **라우팅 정확도**를 올립니다.

- manual_info: 근거 기반(매뉴얼) 답변
- device_control: 기기 제어(설정 변경/시작/중지)
- troubleshooting: 에러코드/증상 진단
- handoff: 안전/권한 이슈로 사람 연결


In [ ]:
class RouteDecision(BaseModel):
    intent: Literal["manual_info","device_control","troubleshooting","handoff"] = Field(description="사용자 의도 분류(제품 탑재형)")

router_llm = llm.with_structured_output(RouteDecision)
# (로컬 llama-server) with_structured_output은 기본적으로 response_format=json_schema로 요청 →
#  llama-server가 스키마를 문법(grammar)으로 변환해 4개 값 중 하나만 나오도록 강제합니다.
# agent화해서 출력 형식 갖추게 할 때 -> create_agent에서 response_format에 ToolStrategy
# 굳이 agent화하지 않을 경우, with_structured_output 메서드로 출력 형식 갖추게 할 수 있음

# Agent Node 우리가 정의해서 쓴다고 한 노드
def router_node(state: GraphState):
    user_text = state["messages"][-1].content
    prompt = """다음 사용자 입력을 제품 탑재형 관점에서 4가지 의도 중 하나로 분류하세요.
- manual_info: 매뉴얼/기능 설명/주의사항 질문
- device_control: 기기 설정 변경/동작 시작/중지 등 제어 요청
- troubleshooting: 에러코드/증상 진단, 해결 절차 안내가 필요한 경우
- handoff: 안전 이슈(연기/과열/가스 냄새), 권한/개인정보 처리 등으로 사람 연결이 필요한 경우

입력:
""" + user_text
    decision = router_llm.invoke(prompt)
    return {"intent": decision.intent}


## 3) Tools 정의

### (A) Manual tool
- `search_manual(device_type, query)` : 더미 매뉴얼 스니펫 검색 + 출처 태그 반환

### (B) Device control tools
- `get_device_status(device)` : 상태 조회(더미)
- `set_device_setting(device, setting, value)` : 설정 변경(더미)

### (C) Troubleshooting tools
- `run_remote_diagnosis(device)` : 원격 진단(더미)
- `create_service_ticket(device, summary)` : 티켓 생성(더미)

> 실제 제품에서는 SDK/API/보안 게이트웨이를 통해 동작하며, 이 노트북은 구조만 체험합니다.


In [ ]:
# --- 더미 매뉴얼/디바이스 데이터 ---
MANUAL_SNIPPETS = {
    "robot_vacuum": {
        "dock": "도킹이 안 되면: 도킹 앞 1m 장애물 제거, 충전단자 먼지 제거, 맵 재설정 고려(초기화 아님).",
        "bump": "충돌이 많으면: 거울/유광 바닥/조명에서 센서 오인식 가능. 청소 구역을 조정해보세요.",
    },
    "washing_machine": {
        "5c": "5C(배수) 에러: 배수 필터 이물질 제거, 배수 호스 꺾임/막힘 확인, 응급 배수 사용. 반복 시 점검 필요.",
    },
    "air_conditioner": {
        "filter": "필터 경고: 필터 세척/건조 후 장착, 필터 리셋 실행(모델별 메뉴 상이).",
        "temp": "권장 온도: 냉방 24~26도(환경에 따라 다름).",
    },
    "refrigerator": {
        "door": "문 열림 경고: 도어 패킹 이물질 제거, 문이 완전히 닫혔는지 확인.",
        "noise": "이상 소음: 수평 불량 시 진동음 증가. 수평 조절 다리 확인.",
    },
}

DEVICES = {
    "vacuum": {"device_type": "robot_vacuum", "status": "docked", "battery": 0.78, "mode": "idle"},
    "ac": {"device_type": "air_conditioner", "status": "online", "mode": "cool", "target_temp_c": 24},
    "washer": {"device_type": "washing_machine", "status": "online", "mode": "standby", "last_error": "5C"},
    "fridge": {"device_type": "refrigerator", "status": "online", "mode": "normal", "door_open": False},
}
TICKETS: List[dict] = []

# --- Manual tool ---
@tool
def search_manual(device_type: Literal["robot_vacuum","washing_machine","air_conditioner","refrigerator"], query: str) -> str:
    """제품 매뉴얼/가이드에서 query와 관련된 스니펫을 반환합니다(더미).
    - 근거 기반 답변이 필요할 때 사용하세요.
    - 반환 문자열에는 [manual:<device_type>:<key>] 출처 태그를 포함합니다.
    """
    q = query.lower()
    entries = MANUAL_SNIPPETS.get(device_type, {})
    hits = []
    for k,v in entries.items():
        if k in q or any(tok in q for tok in ["도킹","충돌","5c","배수","필터","온도","문","소음"]):
            hits.append(f"[manual:{device_type}:{k}] {v}")
    if not hits:
        return f"NO_HIT: {device_type}"
    return "\n".join(hits[:3])

# --- Device tools ---
@tool
def get_device_status(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """기기 상태(더미)를 조회합니다."""
    return json.dumps({"device": device, **DEVICES.get(device, {})}, ensure_ascii=False)

@tool
def set_device_setting(device: Literal["vacuum","ac","washer","fridge"], setting: str, value: str) -> str:
    """기기 설정을 변경합니다(더미). value는 문자열로 전달합니다(예: "22", "cool")."""
    if device not in DEVICES:
        return json.dumps({"ok": False, "error": "unknown_device"}, ensure_ascii=False)
    DEVICES[device][setting] = value
    return json.dumps({"ok": True, "device": device, "updated": {setting: value}}, ensure_ascii=False)

@tool
def run_remote_diagnosis(device: Literal["vacuum","ac","washer","fridge"]) -> str:
    """원격 자가진단(더미)"""
    if device == "washer" and DEVICES[device].get("last_error") == "5C":
        return json.dumps({"ok": True, "device": device, "diagnosis": "drain_issue_likely"}, ensure_ascii=False)
    return json.dumps({"ok": True, "device": device, "diagnosis": "no_critical_issue"}, ensure_ascii=False)

@tool
def create_service_ticket(device: Literal["vacuum","ac","washer","fridge"], summary: str) -> str:
    """서비스 티켓 생성(더미). 개인정보는 포함하지 않습니다."""
    t = {"ticket_id": f"TCK-{1000+len(TICKETS)}", "device": device, "summary": summary[:200]}
    TICKETS.append(t)
    return json.dumps({"ok": True, "ticket": t}, ensure_ascii=False)


## 4) 워크플로우별 Agent 노드

각 intent마다:
- LLM에 tools를 바인딩한 agent 노드
- ToolNode
- tools_condition으로 루프

### 주의
- 한 그래프에서 ToolNode를 3개 쓰기 때문에, 노드/툴 리스트를 분리해서 구성합니다.


In [ ]:
# intent별 tool 세트
manual_tools = [search_manual]
control_tools = [get_device_status, set_device_setting]
troubleshoot_tools = [get_device_status, run_remote_diagnosis, search_manual, create_service_ticket]

manual_llm = llm.bind_tools(manual_tools)
control_llm = llm.bind_tools(control_tools)
troubleshoot_llm = llm.bind_tools(troubleshoot_tools)

def manual_agent_node(state: GraphState):
    sys = (
        "매뉴얼 질문에는 반드시 search_manual 도구를 호출하세요. "
        "최종 답변에는 [manual:<device_type>:<key>] 출처 태그를 문장/항목마다 포함하세요. "
        "근거가 없으면 '매뉴얼에 없음'이라고 답하세요."
    )
    resp = manual_llm.invoke([SystemMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def control_agent_node(state: GraphState):
    sys = (
        "기기 제어 요청에는 get_device_status로 현재 상태를 확인한 뒤, "
        "set_device_setting으로 변경하세요. "
        "불확실하면 실행하지 말고 짧게 확인 질문을 하세요."
    )
    resp = control_llm.invoke([SystemMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def troubleshoot_agent_node(state: GraphState):
    sys = (
        "고장/에러코드 문의에는 get_device_status + run_remote_diagnosis로 근거를 모으고, "
        "필요하면 search_manual로 해결 절차를 찾아 근거 태그와 함께 안내하세요. "
        "반복/위험/해결불가로 판단되면 create_service_ticket을 제안/생성하세요."
    )
    resp = troubleshoot_llm.invoke([SystemMessage(content=sys)] + state["messages"])
    return {"messages": [resp]}

def handoff_node(state: GraphState):
    # handoff는 도구 없이 바로 종료
    msg = (
        "안전/권한/개인정보 이슈가 있어 바로 조치하기 어렵습니다. "
        "안전을 위해 전원을 차단하고(가능하면), 사람 상담/기사 연결로 안내드릴게요."
    )
    return {"messages": [AIMessage(content=msg)]}  # 시스템(어시스턴트) 응답

manual_tool_node = ToolNode(manual_tools)
control_tool_node = ToolNode(control_tools)
troubleshoot_tool_node = ToolNode(troubleshoot_tools)


## 5) Graph 구성 (Router → 서브플로우)

- START → router
- router 결과(intent)에 따라 각각의 서브플로우로 진입
- 각 서브플로우는 tools_condition으로 반복
- 최종은 END

또한 checkpointer를 붙여 thread_id 기반 메모리를 켭니다.


In [ ]:
builder = StateGraph(GraphState)
builder.add_node("router", router_node)
builder.add_node("manual_agent", manual_agent_node)
builder.add_node("manual_tools", manual_tool_node)
builder.add_node("control_agent", control_agent_node)
builder.add_node("control_tools", control_tool_node)
builder.add_node("troubleshoot_agent", troubleshoot_agent_node)
builder.add_node("troubleshoot_tools", troubleshoot_tool_node)
builder.add_node("handoff", handoff_node)
# START → router
builder.add_edge(START, "router")
def route_from_intent(state: GraphState):
    return state.get("intent") or "manual_info"  # fallback
builder.add_conditional_edges(
    "router",
    route_from_intent,
    {
        "manual_info": "manual_agent",
        "device_control": "control_agent",
        "troubleshooting": "troubleshoot_agent",
        "handoff": "handoff",
    }
)
# manual flow
builder.add_conditional_edges("manual_agent", tools_condition, {"tools": "manual_tools", "__end__": END})
builder.add_edge("manual_tools", "manual_agent")
# control flow
builder.add_conditional_edges("control_agent", tools_condition, {"tools": "control_tools", "__end__": END})
builder.add_edge("control_tools", "control_agent")
# troubleshoot flow
builder.add_conditional_edges("troubleshoot_agent", tools_condition, {"tools": "troubleshoot_tools", "__end__": END})
builder.add_edge("troubleshoot_tools", "troubleshoot_agent")
# handoff flow (바로 종료)
builder.add_edge("handoff", END)
graph = builder.compile(checkpointer=InMemorySaver())


In [ ]:
from IPython.display import display, Image

# draw_mermaid_png()는 기본적으로 mermaid.ink 웹 API를 사용 → 인터넷이 없으면 실패할 수 있음
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"PNG 렌더링 실패({type(e).__name__}) → mermaid 텍스트로 출력합니다 (https://mermaid.live 에 붙여넣기)")
    print(graph.get_graph().draw_mermaid())


## 6) Streaming으로 실행 과정 관찰

노드별 업데이트를 보면서 Tool 호출이 어떤 흐름으로 일어나는지 확인합니다.


In [ ]:
def _stream_graph_impl(user_input: str, thread_id: str = "t1"):
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 25}  # 로컬 모델 tool 루프 상한
    for event in graph.stream({"messages": [HumanMessage(content=user_input)], "intent": None}, config=config, stream_mode="updates"):
        print("\n--- event ---")
        for node, payload in event.items():
            print("node:", node)
            if isinstance(payload, dict) and "messages" in payload and payload["messages"]:
                msg = payload["messages"][-1]
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print(" tool_calls:", [tc["name"] for tc in msg.tool_calls])
                else:
                    print(" text:", str(msg.content)[:120])

def stream_graph(user_input: str, thread_id: str = "t1"):
    """recursion_limit 초과 시에도 셀이 멈추지 않도록 감싼 버전"""
    from langgraph.errors import GraphRecursionError
    try:
        _stream_graph_impl(user_input, thread_id)
    except GraphRecursionError:
        print("⚠ recursion_limit 초과 — 로컬 모델이 tool 호출을 반복했습니다. (노드 프롬프트 개선 포인트)")

# 예시
stream_graph("세탁기 5C 에러가 떠요. 어떻게 해야 하죠?") 


### 🔍 네 가지 경로 모두 관찰하기

입력에 따라 router가 다른 경로를 고르는지 확인해 봅시다. `node:` 출력 순서로 그래프에서 어떤 경로를 지났는지 따라가 보세요.

In [ ]:
demo_inputs = {
    "manual_info":     "로봇청소기 도킹이 안 될 때 매뉴얼에는 뭐라고 나와 있어?",
    "device_control":  "거실 에어컨 온도를 22도로 바꿔줘",
    "troubleshooting": "세탁기에 5C 에러가 계속 떠요. 필터랑 호스는 이미 확인했어요",
    "handoff":         "냉장고 뒤에서 타는 냄새가 나고 연기가 나요",
}
for expected, text in demo_inputs.items():
    print(f"\n==================== 기대 경로: {expected}\n입력: {text}")
    stream_graph(text, thread_id=f"demo-{expected}")

## 📝 실습 과제

> 아래 코드 셀의 TODO 항목을 구현하세요.


In [ ]:
# ✅ TODO 셀 (학생용)
# 1) Router를 업그레이드:
#    - 안전 키워드(연기/과열/가스/불꽃 등) 포함 시 handoff로 강제
# 2) device_control에서 '확인 단계'를 추가:
#    - 예: 'factory_reset' 같은 위험 작업은 별도 confirm 노드를 거쳐야만 tool 호출 가능하도록 그래프 확장
# 3) troubleshooting에서 티켓 생성 조건을 강화:
#    - 진단 결과가 drain_issue_likely 이고 사용자가 이미 필터/호스 확인했다고 하면 create_service_ticket 호출

raise NotImplementedError("학생용 TODO 셀입니다. 구현 후 stream_graph로 동작 확인해보세요.")


In [ ]:
# ✅ 참고 답안 (핵심만) — 막힐 때만 참고하세요
from langchain_core.messages import AIMessage
from langgraph.types import interrupt, Command

# --- TODO 1) Router: 안전 키워드 → LLM 우회 즉시 handoff ---
SAFETY_KEYWORDS = ["연기", "과열", "가스", "불꽃", "화재", "타는 냄새", "누전"]

def router_node_v2(state):
    text = str(state["messages"][-1].content)
    if any(kw in text for kw in SAFETY_KEYWORDS):
        return {"intent": "handoff"}          # 안전 케이스는 분류기를 건너뜀(오분류/지연 방지)
    return {"intent": router_llm.invoke(text).intent}   # 그 외엔 기존 LLM 분류


# --- TODO 2) device_control: 위험 작업 confirm 노드(interrupt 로 사람 승인) ---
DANGEROUS_SETTINGS = {"factory_reset", "firmware_update"}

def control_route_after_agent(state):
    last = state["messages"][-1]
    if not getattr(last, "tool_calls", None):
        return END
    danger = any(tc.get("name") == "set_device_setting"
                 and tc.get("args", {}).get("setting") in DANGEROUS_SETTINGS
                 for tc in last.tool_calls)
    return "confirm" if danger else "tools"

def confirm_node(state):
    # interrupt() 가 그래프를 멈추고 호출자에게 값을 넘김 →
    # 호출자는 Command(resume="approve"/"reject") 로 재개
    decision = interrupt({"prompt": "위험 작업입니다. approve / reject?"})
    return {"confirmed": str(decision).lower().startswith("approve")}

def control_route_after_confirm(state):
    return "tools" if state.get("confirmed") else END

# 그래프 배선(요지):
#   control_agent -[control_route_after_agent]-> {confirm, control_tools, END}
#   confirm       -[control_route_after_confirm]-> {control_tools, END}
#   (State 에 confirmed: Optional[bool] 필드 추가)


# --- TODO 3) troubleshooting: 배수 이슈 + 사용자 점검 완료 → 자동 티켓 ---
def _latest_diagnosis(state):
    for m in reversed(state["messages"]):
        if isinstance(m, ToolMessage) and getattr(m, "name", "") == "run_remote_diagnosis":
            try:
                return json.loads(m.content)
            except Exception:
                return None
    return None

def _user_already_self_checked(state):
    txt = " ".join(str(m.content) for m in state["messages"] if isinstance(m, HumanMessage))
    return any(t in txt for t in ("필터", "배수", "호스")) and any(w in txt for w in ("확인", "점검", "했"))

def _already_created_ticket(state):  # 중복 자동 호출 방지(idempotency)
    return any(isinstance(m, ToolMessage) and getattr(m, "name", "") == "create_service_ticket"
               for m in state["messages"])

def troubleshoot_agent_node_v2(state):
    diag = _latest_diagnosis(state)
    if (diag and diag.get("diagnosis") == "drain_issue_likely"
            and _user_already_self_checked(state) and not _already_created_ticket(state)):
        # LLM 판단을 기다리지 않고 티켓 tool_call 을 직접 생성
        tc = {"id": "auto_ticket", "name": "create_service_ticket",
              "args": {"device": diag.get("device", "washer"),
                       "summary": "배수 이슈 의심, 사용자 점검 완료, 현장 점검 필요"}}
        return {"messages": [AIMessage(content="", tool_calls=[tc])]}
    return troubleshoot_agent_node(state)   # 그 외에는 기존 노드 로직 재사용

# 위 노드들을 GraphStateV2(confirmed 필드 추가)로 다시 빌드하면 v2 그래프가 완성됩니다.

## ✅ 정리 & 스스로 점검

**정리**
- LangGraph는 **State + Node + (조건부) Edge**로 LLM 앱의 흐름을 명시적으로 설계합니다.
- 분류(router)는 구조화 출력으로, 각 경로는 전용 도구 세트를 가진 작은 에이전트로 구성합니다.
- 안전·확인 같은 **반드시 지켜야 하는 규칙은 LLM이 아니라 그래프 구조(코드)로** 강제합니다 (TODO 1·2).

**05~09 한눈에 보기**

| 실습 | 핵심 기술 | 작은 로컬 모델을 보완하는 방법 |
|---|---|---|
| 05 | 구조화 출력 | 스키마·grammar로 형식 강제, validator로 내용 검사 |
| 06 | 에이전트 | 명확한 도구 설명, 권한은 코드로 |
| 07 | RAG | 근거 검색으로 지식 보충, 출처 강제 |
| 08 | 멀티에이전트 | 역할 분리, 계산은 코드로 |
| 09 | LangGraph | 흐름을 그래프로 고정, 위험 경로 강제 |

**스스로 점검**
1. "연기가 나요" 입력이 handoff로 가지 않았다면, 어디를 고치는 게 가장 확실할까요? (TODO 1)
2. control 경로에서 위험한 설정 변경을 막는 가장 안전한 방법은? (TODO 2)
3. 같은 thread_id로 두 번째 질문을 하면 그래프는 무엇을 기억하고 있을까요?

**다음 단계 →** 10번부터는 LLM 서버가 필요 없습니다. VRAM 확보를 위해 `python local_server.py stop`를 권장합니다.